# 1. Переходим в папку проекта
cd C:\LABS\AutoAnnotationProj

# 2. Разрешаем скрипты только для этого окна
Set-ExecutionPolicy -Scope Process -ExecutionPolicy Bypass

# 3. Создаём виртуальное окружение (если ещё не создано)
py -3.12 -m venv venv

# 4. Активируем окружение
.\venv\Scripts\Activate.ps1

# Теперь в этом же окне PowerShell установи ultralytics:
pip install --upgrade pip
pip install ultralytics==8.3.0

# Проверь:
python -c "import ultralytics; print(ultralytics.__version__)"

# Обновляем до самой свежей версии (8.4.33 на апрель 2026)
pip install -U ultralytics

# Устанавливаем Torch с CUDA специально под NDIDIA
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# В случае ошибок с ipykernel
c:/LABS/AutoAnnotationProj/venv/Scripts/python.exe -m pip install ipykernel -U --force-reinstall


pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126



In [3]:
import torch
print(torch.cuda.is_available()) # Should return True
print(torch.version.cuda)         # Displays the CUDA version PyTorch is using

True
12.6


In [ ]:
import torch
from ultralytics import YOLO
import os

# Проверка GPU
print("CUDA доступен:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM всего:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "ГБ")

# Загружаем модель (m — оптимальный баланс качество / скорость / VRAM 8 ГБ)
model = YOLO("yolov8m-seg.pt")        # Если будет мало VRAM — замени на yolov8s-seg.pt

# Путь к data.yaml
data_path = r"C:\LABS\AutoAnnotationProj\coronary_dataset\data.yaml"

# === ОБУЧЕНИЕ ===
results = model.train(
    data=data_path,
    epochs=100,                    # можно 50–150, потом дообучить
    imgsz=1024,                    # для КТ-ангио обычно 1024 или 640
    batch=4,                       # под 8 ГБ VRAM идеально (можно попробовать 6)
    device=0,                      # 0 = твоя RTX 4060 Ti
    patience=30,                   # early stopping
    name="coronary_yolov8m_seg",
    exist_ok=True,
    pretrained=True,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    augment=True,
    cache=True,
    verbose=True,
    plots=True
)

print("\n✅ Обучение завершено!")
print("Лучшие веса лежат здесь:")
print(results.save_dir / "weights/best.pt")

CUDA доступен: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM всего: 8.0 ГБ
Ultralytics 8.4.33  Python-3.12.9 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\LABS\AutoAnnotationProj\coronary_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coro